# 07 — Análisis Exploratorio
## Consultas sobre la Capa Gold

Notebook de exploración ad-hoc sobre los datos de Gold.
Lee directamente los Parquet del modelo dimensional y calcula KPIs.

**Requiere:** Haber ejecutado los notebooks 04, 05 y 06 (Gold)


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.paths import GOLD, str_path
from src.spark_utils import get_spark

spark = get_spark(app_name="MEF_Analisis", memory="4g")
print("Listo para análisis")

[OK] SparkSession lista | version=3.5.0 | master=local[*]
Listo para análisis


## 1. Carga del Modelo Dimensional

In [2]:
def load_gold(key):
    path = str_path(GOLD[key])
    df = spark.read.parquet(path)
    n = df.count()
    print(f"  {key:<35} {n:>10,} filas | {len(df.columns)} cols")
    return df

print("Cargando Gold SIAF:")
fact    = load_gold("fact_ejecucion_presupuestal")
d_tiempo = load_gold("dim_tiempo")
d_geo    = load_gold("dim_geografia")
d_muni   = load_gold("dim_municipalidad")
d_clas   = load_gold("dim_clasificacion_ingreso")
d_fin    = load_gold("dim_financiamiento")

Cargando Gold SIAF:
  fact_ejecucion_presupuestal          8,999,223 filas | 13 cols
  dim_tiempo                                 174 filas | 6 cols
  dim_geografia                            1,896 filas | 8 cols
  dim_municipalidad                        1,964 filas | 9 cols
  dim_clasificacion_ingreso                  423 filas | 11 cols
  dim_financiamiento                         145 filas | 7 cols


## 2. KPI: Ejecución Presupuestal por Año

In [3]:
print("Ejecución Presupuestal Total por Año (en millones de S/.):\n")
(
    fact.join(d_tiempo, "SK_Tiempo")
    .groupBy("Ano")
    .agg(
        F.sum("Monto_PIM").alias("PIM_Total"),
        F.sum("Monto_Recaudado").alias("Recaudado_Total"),
        F.count("*").alias("N_Registros")
    )
    .withColumn("Tasa_Avance_%", F.round(F.col("Recaudado_Total") / F.col("PIM_Total") * 100, 2))
    .withColumn("PIM_MM", F.round(F.col("PIM_Total") / 1_000_000, 1))
    .withColumn("Recaudado_MM", F.round(F.col("Recaudado_Total") / 1_000_000, 1))
    .select("Ano", "PIM_MM", "Recaudado_MM", "Tasa_Avance_%", "N_Registros")
    .orderBy("Ano")
    .show()
)

Ejecución Presupuestal Total por Año (en millones de S/.):

+----+-------+------------+-------------+-----------+
| Ano| PIM_MM|Recaudado_MM|Tasa_Avance_%|N_Registros|
+----+-------+------------+-------------+-----------+
|2012|27217.4|     26288.4|        96.59|     638925|
|2013|28965.6|     26577.7|        91.76|     590349|
|2014|27088.6|     25872.5|        95.51|     611717|
|2015|24004.5|     23399.8|        97.48|     632600|
|2016|26468.3|     25979.1|        98.15|     613538|
|2017|29127.8|     28884.2|        99.16|     550948|
|2018|33802.1|     32427.3|        95.93|     527741|
|2019|30619.3|     29214.2|        95.41|     571750|
|2020|34944.3|     33468.2|        95.78|     572621|
|2021|41981.2|     41024.8|        97.72|     691765|
|2022|46232.5|     45732.3|        98.92|     638548|
|2023|40030.5|     42563.6|       106.33|     630891|
|2024|42050.6|     44948.5|       106.89|     643442|
|2025|43231.2|     46171.8|       106.80|     739985|
|2026|41237.3|     261

## 3. KPI: Top 10 Departamentos por Tasa de Avance

In [4]:
print("Top 10 Departamentos por Tasa de Avance (último año disponible):\n")

ultimo_ano = fact.join(d_tiempo, "SK_Tiempo").agg(F.max("Ano")).collect()[0][0]
print(f"  Año de análisis: {ultimo_ano}\n")

(
    fact.join(d_tiempo, "SK_Tiempo").filter(F.col("Ano") == ultimo_ano)
    .join(d_geo, "SK_Geografia")
    .groupBy("Nombre_Departamento")
    .agg(
        F.sum("Monto_PIM").alias("PIM"),
        F.sum("Monto_Recaudado").alias("Recaudado"),
        F.countDistinct("SK_Municipalidad").alias("N_Municipalidades")
    )
    .withColumn("Tasa_%", F.round(F.col("Recaudado") / F.col("PIM") * 100, 2))
    .orderBy(F.col("Tasa_%").desc())
    .show(10, truncate=False)
)

Top 10 Departamentos por Tasa de Avance (último año disponible):

  Año de análisis: 2026

+-------------------+-------------+-------------+-----------------+------+
|Nombre_Departamento|PIM          |Recaudado    |N_Municipalidades|Tasa_%|
+-------------------+-------------+-------------+-----------------+------+
|LIMA               |9119530377.00|7920985015.90|173              |86.86 |
|AREQUIPA           |2518730517.00|2015032560.72|112              |80.00 |
|PASCO              |391229482.00 |274210862.86 |30               |70.09 |
|MOQUEGUA           |1039056116.00|722832236.48 |21               |69.57 |
|APURIMAC           |1051379244.00|703309485.23 |85               |66.89 |
|ICA                |1956352699.00|1298008299.01|44               |66.35 |
|LA LIBERTAD        |1939867833.00|1243449996.44|84               |64.10 |
|JUNIN              |1381723995.00|869751512.18 |126              |62.95 |
|CAJAMARCA          |1737944612.00|1057137628.27|127              |60.83 |
|MADRE DE

## 4. KPI: Distribución de Calidad de Datos

In [5]:
print("Distribución SK_Calidad:")
calidad_map = {
    1: "Limpio",
    2: "Advertencias menores",
    3: "Inconsistencias de catálogo",
    4: "Error crítico"
}
total = fact.count()
(
    fact.groupBy("SK_Calidad")
    .count()
    .withColumn("Pct", F.round(F.col("count") / total * 100, 2))
    .orderBy("SK_Calidad")
    .show()
)
print(f"  Total registros Fact: {total:,}")

Distribución SK_Calidad:
+----------+-------+-----+
|SK_Calidad|  count|  Pct|
+----------+-------+-----+
|         1|8896130|98.85|
|         2|  89545|  1.0|
|         4|  13548| 0.15|
+----------+-------+-----+

  Total registros Fact: 8,999,223


## 5. KPI: Top Fuentes de Financiamiento

In [6]:
print("PIM por Fuente de Financiamiento (todos los años):\n")
(
    fact.join(d_fin, "SK_Financiamiento")
    .groupBy("Fuente_Financiamiento")
    .agg(
        F.round(F.sum("Monto_PIM") / 1_000_000, 1).alias("PIM_MM"),
        F.round(F.sum("Monto_Recaudado") / 1_000_000, 1).alias("Recaudado_MM")
    )
    .orderBy(F.col("PIM_MM").desc())
    .show(10, truncate=False)
)

PIM por Fuente de Financiamiento (todos los años):

+---------------------------------------------+--------+------------+
|Fuente_Financiamiento                        |PIM_MM  |Recaudado_MM|
+---------------------------------------------+--------+------------+
|RECURSOS DETERMINADOS                        |368153.3|352917.9    |
|RECURSOS DIRECTAMENTE RECAUDADOS             |63749.6 |60190.7     |
|RECURSOS POR OPERACIONES OFICIALES DE CREDITO|59501.0 |59099.9     |
|DONACIONES Y TRANSFERENCIAS                  |25597.2 |26447.2     |
+---------------------------------------------+--------+------------+



## 6. SISMEPRE — Vista Rápida

In [7]:
from src.paths import GOLD

sismepre_keys = [
    "sismepre_formulario", "sismepre_preguntas", "sismepre_respuestas",
    "sismepre_estadistica", "sismepre_esat_atm",
    "sismepre_entidad_estado", "sismepre_ano_aplicacion"
]

print("SISMEPRE Gold Summary:\n")
for key in sismepre_keys:
    p = GOLD[key]
    if p.exists() and any(p.rglob("*.parquet")):
        df_s = spark.read.parquet(str_path(p))
        print(f"  {key:<35} {df_s.count():>8,} filas | {len(df_s.columns)} cols")
    else:
        print(f"  {key:<35} sin datos")

SISMEPRE Gold Summary:

  sismepre_formulario                       98 filas | 11 cols
  sismepre_preguntas                       836 filas | 16 cols
  sismepre_respuestas                  174,210 filas | 13 cols
  sismepre_estadistica                     233 filas | 8 cols
  sismepre_esat_atm                    134,170 filas | 44 cols
  sismepre_entidad_estado               19,037 filas | 15 cols
  sismepre_ano_aplicacion                   26 filas | 9 cols
